# 랜딩 페이지 A/B 테스트 분석
**[통계적 검정과 인과추론] 종합 과제**

- 데이터 출처: Kaggle, *A/B Testing* (zhangluyuan) — https://www.kaggle.com/datasets/zhangluyuan/ab-testing
- 내려받은 날짜: 2026-11-06  <!-- TODO: 실제 내려받은 날짜가 다르면 수정 -->
- 데이터 파일: `ab_data.csv` (컬럼: user_id, timestamp, group, landing_page, converted)

**분석 흐름**: 0. 준비 → 1. 데이터 점검 → 2. 질문과 가설 → 3. 검정 선택 → 4. 검정·보고 → 5. 한 걸음 더 → 6. 의사결정

## 0. 준비 — 라이브러리 불러오기

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# 재현성을 위해 난수 시드 고정 (5단계에서 난수를 쓸 수도 있으므로 미리 고정)
SEED = 42
np.random.seed(SEED)

plt.rcParams['font.family'] = 'Malgun Gothic'   # 그래프 한글 깨짐 방지 (Windows)
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 점검
검정보다 먼저 합니다. 데이터가 오염돼 있으면 p값은 의미가 없습니다.

1-1 기본 확인 → 1-2 배정 일관성 → 1-3 중복 → 1-4 정제 → 1-5 SRM 점검

> ⚠️ 이 단계에서는 **그룹별 전환율을 보지 않습니다.** 가설(2단계)을 먼저 정해야 하기 때문입니다.

### 1-1. 기본 확인 — 행 수, 컬럼, 자료형, 결측치

In [2]:
df_raw = pd.read_csv('ab_data.csv')
print('행 수, 열 수:', df_raw.shape)
df_raw.head()

행 수, 열 수: (294478, 5)


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [3]:
df_raw.info()        # 컬럼별 자료형과 non-null 개수

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [4]:
df_raw.isnull().sum()   # 컬럼별 결측치 개수

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [5]:
# 범주형 컬럼에 어떤 값들이 있는지 확인
for col in ['group', 'landing_page', 'converted']:
    print(df_raw[col].value_counts(), '\n')

# 실험 기간
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
print('실험 기간:', df_raw['timestamp'].min(), '~', df_raw['timestamp'].max())

group
treatment    147276
control      147202
Name: count, dtype: int64 

landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64 

converted
0    259241
1     35237
Name: count, dtype: int64 

실험 기간: 2017-01-02 13:42:05.378582 ~ 2017-01-24 13:41:54.460509


### 1-2. 배정 일관성 — 처치 라벨이 어긋난 행이 있는가?
정상이라면 `control ↔ old_page`, `treatment ↔ new_page` 조합만 있어야 합니다.

In [6]:
pd.crosstab(df_raw['group'], df_raw['landing_page'])

landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


In [7]:
mismatch = ((df_raw['group'] == 'control')   & (df_raw['landing_page'] == 'new_page')) | \
           ((df_raw['group'] == 'treatment') & (df_raw['landing_page'] == 'old_page'))
print('배정이 어긋난 행 수:', mismatch.sum())

배정이 어긋난 행 수: 3893


### 1-3. 중복 확인 — 같은 user_id가 두 번 이상 나오는가?
한 사용자는 한 번만 실험에 들어가야 합니다(관측치의 독립성).

In [8]:
print('전체 행 수     :', len(df_raw))
print('고유 user_id 수:', df_raw['user_id'].nunique())
print('중복된 user_id 수:', df_raw['user_id'].duplicated().sum())

전체 행 수     : 294478
고유 user_id 수: 290584
중복된 user_id 수: 3894


In [9]:
# 배정이 정상인 행만 남긴 뒤에도 중복이 남는지 확인
df_ok = df_raw[~mismatch]
dup_ids = df_ok.loc[df_ok['user_id'].duplicated(keep=False), 'user_id'].unique()
print('정상 배정 행 중 중복 user_id:', len(dup_ids))
df_ok[df_ok['user_id'].isin(dup_ids)].sort_values('user_id')

정상 배정 행 중 중복 user_id: 1


,user_id,timestamp,group,landing_page,converted
1899,773192,2017-01-09 05:37:58.781806,treatment,new_page,0
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0


### 1-4. 정제
**정제 규칙** (TODO: 1-2, 1-3 결과를 보고 이유를 한 줄씩 직접 적어 보세요)
- 배정이 어긋난 행 제거 — 이유:
- 중복 user_id는 첫 기록만 남김 — 이유:

In [10]:
df = df_ok.drop_duplicates(subset='user_id', keep='first').copy()

print(f'정제 전: {len(df_raw):,}행')
print(f'배정 불일치 제거 후: {len(df_ok):,}행  (-{mismatch.sum():,})')
print(f'중복 제거 후(최종): {len(df):,}행  (-{len(df_ok) - len(df):,})')
assert df['user_id'].is_unique

정제 전: 294,478행
배정 불일치 제거 후: 290,585행  (-3,893)
중복 제거 후(최종): 290,584행  (-1)


### 1-5. SRM 점검 — 배정 비율이 50:50인가?
⚠️ 여기서는 **p가 커야 정상**입니다(= 의도한 50:50과 다르지 않다). 평소 검정과 해석 방향이 반대입니다.

In [11]:
n_a = (df['group'] == 'control').sum()
n_b = (df['group'] == 'treatment').sum()
print(f'control: {n_a:,} ({n_a/(n_a+n_b):.2%}),  treatment: {n_b:,} ({n_b/(n_a+n_b):.2%})')

srm = stats.chisquare([n_a, n_b], [(n_a + n_b) / 2] * 2)
print(f'SRM 카이제곱 = {srm.statistic:.3f}, p = {srm.pvalue:.4f}')

control: 145,274 (49.99%),  treatment: 145,310 (50.01%)
SRM 카이제곱 = 0.004, p = 0.9468


**1단계 정리** (TODO: 직접 작성)
- 정제 전 → 후 행 수:
- SRM 결과 해석:

## 2. 질문과 가설 정하기
⚠️ **결과를 보기 전에** 정하고, 끝까지 바꾸지 않습니다.

- **질문 (한 문장)**: TODO  (예: "새 랜딩 페이지로 바꾸면 전환율이 올라갈까?")
- **H0 (귀무가설)**: TODO
- **H1 (대립가설)**: TODO
- **유의수준 α**: TODO
- **단측 / 양측**: TODO — 선택한 이유:

In [12]:
ALPHA = None          # TODO: 2단계에서 정한 유의수준 (예: 0.05)
ALTERNATIVE = None    # TODO: 'two-sided'(양측) 또는 'larger'(새 페이지가 더 크다, 단측)

## 3. 검정 고르기 ⭐
아래 세 질문에 답해 보세요.
1. 성공 지표(`converted`)는 비율인가, 연속형인가? → TODO
2. 비교할 집단은 몇 개인가? → TODO
3. (연속형일 때만) 정규성·등분산 가정은? → TODO

**나는 ______ 검정을 골랐다. 왜냐하면 …** (TODO: 한 문단. 코드보다 중요합니다!)

## 4. 검정 수행과 3종 세트 보고
① 점추정(차이) ② 95% 신뢰구간 ③ 효과 크기를 함께 적습니다. 비율이면 절대 차이(%p)와 상대 차이(%)를 둘 다 적습니다.

### 4-1. 그룹별 전환율 (요약)

In [13]:
# TODO: 2단계 가설을 정한 뒤에 실행합니다

### 4-2. 검정 수행

In [14]:
# TODO

### 4-3. 점추정 · 95% 신뢰구간 · 효과 크기

In [15]:
# TODO

**보고 문장** (TODO)
> 예시 형식: 새 페이지의 전환율은 __% vs __% (차이 __%p, 상대 __%), 95% CI [__, __], χ²(1) = __, p = __, 효과 크기 = __

## 5. 한 걸음 더 — 1개 선택
- A. 검정력과 MDE (이 실험은 효과를 잡을 힘이 있었나?)
- B. 가정이 깨졌다면 (연속형 지표용 — 이 데이터엔 잘 맞지 않음)
- C. 세그먼트(요일·시간대) 반복 검정 + 다중검정 보정
- D. 비용까지 따져 보기

**선택: TODO** — 이유:

In [16]:
# TODO

## 6. 의사결정
1. **질문과 가설** — TODO
2. **데이터 위생 점검** — TODO (정제 전후 행 수 포함)
3. **검정 선택 근거** — TODO
4. **결과** — 점추정 · 신뢰구간 · 효과 크기: TODO
5. **의사결정** — 출시 / 보류 / 데이터 추가 수집 중 하나 + 근거: TODO
6. **한계** — 이 결론이 기대는 가정, 이 데이터로 알 수 없는 것: TODO